In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os
from langchain_core.tools import tool
from pydantic import BaseModel
from typing import Annotated
from langchain_core.messages import AnyMessage, HumanMessage
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
import subprocess
from rich.console import Console
from rich.panel import Panel
from rich.pretty import Pretty

console = Console()

def show(response):
    console.print(
        Panel(
            Pretty(response),
            title="🤖 LLM Response",
            border_style="green",
            expand=False
        )
    )

def show_markdown(response):
    Console().print(Markdown(response))

load_dotenv()

# llm = ChatOpenAI(
#     model="stepfun/step-3.5-flash:free",
#     api_key=os.getenv("OPENROUTER_API_KEY"),
#     base_url="https://openrouter.ai/api/v1",
# )

llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

# llm = ChatOpenAI(
#     model="openai/gpt-5-nano",
#     api_key=os.getenv("OPENROUTER_API_KEY"),
#     base_url="https://openrouter.ai/api/v1",
# )


# llm = ChatOpenAI(
#     model="gemini-2.5-flash",
#     api_key=os.getenv("GEMINI_API_KEY"),
#     base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
# )

In [ ]:
@tool
def bash(command: str) -> str:
    """Executes a bash command and returns the output or error."""

    BASH_PATH = r"C:\Program Files\Git\bin\bash.exe" 
    try:
        # Use shell=True to allow piping and redirections
        result = subprocess.run(
            [BASH_PATH, "-c", command],  # Use the specific bash executable
            capture_output=True, 
            text=True, 
            timeout=30
        )
        return result.stdout if result.returncode == 0 else result.stderr
    except subprocess.TimeoutExpired:
        return "Error: Command timed out after 30 seconds."
    except FileNotFoundError:
        return "Error: Git Bash not found at the specified path."
    except Exception as e:
        return f"Unexpected Error: {str(e)}"

@tool
def read_file(path: str) -> str:
    """Reads and returns the content of a file at the specified path."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: The file at {path} was not found."
    except Exception as e:
        return f"Error reading file: {str(e)}"

@tool
def write_file(path: str, content: str) -> str:
    """Writes content to a file, creating parent directories if they don't exist."""
    try:
        file_path = Path(path)
        # Create directories if they are missing (similar to 'mkdir -p')
        file_path.parent.mkdir(parents=True, exist_ok=True)
        
        file_path.write_text(content, encoding='utf-8')
        return f"Successfully wrote {len(content)} characters to {path}"
    except Exception as e:
        return f"Error writing to {path}: {str(e)}"

In [70]:
class agentState(BaseModel):
    messages: Annotated[list[AnyMessage], operator.add]

In [71]:
class graphAgent():
    def __init__(self, session_id: Optional[str] = None):
        self.llm = ChatOpenAI(
            model="openai/gpt-oss-120b",
            api_key=os.getenv("OPENROUTER_API_KEY"),
            base_url="https://openrouter.ai/api/v1",
        )
        self.tools = [bash, read_file, write_file]
        self.llm_with_tools = self.llm.bind_tools(self.tools)
        self.agent = self._build_graph()

    def _build_graph(self):
        def agent_node(state: agentState):
            response = self.llm_with_tools.invoke(state.messages)
            return {'messages': [response]}

        graph = StateGraph(agentState)
        graph.add_node('tools', ToolNode(self.tools))
        graph.add_node('agent', agent_node)
        graph.set_entry_point('agent')
        graph.add_conditional_edges(
            'agent',
            tools_condition
        )
        graph.add_edge('tools', 'agent')
        app = graph.compile()
        return app
    
    def run(self, user_input: str) -> str:
        result = self.agent.invoke({
            'messages': [HumanMessage(content=user_input)]
        })
        return {
            'result': result['messages'][-1].content
        }

    def stream(self, user_input: str) -> str:
        for chunk in self.agent.stream(
            agentState(
                messages = [HumanMessage(content=user_input)]
            ),
            stream_mode="updates"
        ):
            show(chunk)

In [73]:
agent = graphAgent()
agent.stream("Read the other file in current directory")

╭────────────────────────────── 🤖 LLM Response ──────────────────────────────╮
│ {                                                                           │
│     'agent': {                                                              │
│         'messages': [                                                       │
│             AIMessage(                                                      │
│                 content='',                                                 │
│                 additional_kwargs={'refusal': None},                        │
│                 response_metadata={                                         │
│                     'token_usage': {                                        │
│                         'completion_tokens': 108,                           │
│                         'prompt_tokens': 191,                               │
│                         'total_tokens': 299,                                │
│                         'completion_tokens_details': {                      │
│                             'accepted_prediction_tokens': None,             │
│                             'audio_tokens': 0,                              │
│                             'reasoning_tokens': 91,                         │
│                             'rejected_prediction_tokens': None,             │
│                             'image_tokens': 0                               │
│                         },                                                  │
│                         'prompt_tokens_details': {                          │
│                             'audio_tokens': 0,                              │
│                             'cached_tokens': 64,                            │
│                             'cache_write_tokens': 0,                        │
│                             'video_tokens': 0                               │
│                         },                                                  │
│                         'cost': 9.345e-05,                                  │
│                         'is_byok': False,                                   │
│                         'cost_details': {                                   │
│                             'upstream_inference_cost': 9.345e-05,           │
│                             'upstream_inference_prompt_cost': 2.865e-05,    │
│                             'upstream_inference_completions_cost': 6.48e-05 │
│                         }                                                   │
│                     },                                                      │
│                     'model_provider': 'openai',                             │
│                     'model_name': 'openai/gpt-oss-120b',                    │
│                     'system_fingerprint': None,                             │
│                     'id': 'gen-1774640365-W76bzjl4kccYRzI1tqXM',            │
│                     'finish_reason': 'tool_calls',                          │
│                     'logprobs': None                                        │
│                 },                                                          │
│                 id='lc_run--019d30ce-fd69-74e2-931e-a88b2a9cad5c-0',        │
│                 tool_calls=[                                                │
│                     {                                                       │
│                         'name': 'bash',                                     │
│                         'args': {'command': 'ls -a'},                       │
│                         'id': 'chatcmpl-tool-bd3eb929de0170ff',             │
│                         'type': 'tool_call'                                 │
│                     }                                                       │
│                 ],                                                          │
│                 invalid_tool_calls=[],                                      │
│

╭────────────────────────────── 🤖 LLM Response ──────────────────────────────╮
│ {                                                                           │
│     'tools': {                                                              │
│         'messages': [                                                       │
│             ToolMessage(                                                    │
│                 content='.\n..\nagent\nlogs\nmain.py\ntest.ipynb\ntests\n', │
│                 name='bash',                                                │
│                 tool_call_id='chatcmpl-tool-bd3eb929de0170ff'               │
│             )                                                               │
│         ]                                                                   │
│     }                                                                       │
│ }                                                                           │
╰─────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 🤖 LLM Response ──────────────────────────────╮
│ {                                                                           │
│     'agent': {                                                              │
│         'messages': [                                                       │
│             AIMessage(                                                      │
│                 content='',                                                 │
│                 additional_kwargs={'refusal': None},                        │
│                 response_metadata={                                         │
│                     'token_usage': {                                        │
│                         'completion_tokens': 57,                            │
│                         'prompt_tokens': 240,                               │
│                         'total_tokens': 297,                                │
│                         'completion_tokens_details': {                      │
│                             'accepted_prediction_tokens': None,             │
│                             'audio_tokens': 0,                              │
│                             'reasoning_tokens': 38,                         │
│                             'rejected_prediction_tokens': None,             │
│                             'image_tokens': 0                               │
│                         },                                                  │
│                         'prompt_tokens_details': {                          │
│                             'audio_tokens': 0,                              │
│                             'cached_tokens': 64,                            │
│                             'cache_write_tokens': 0,                        │
│                             'video_tokens': 0                               │
│                         },                                                  │
│                         'cost': 7.02e-05,                                   │
│                         'is_byok': False,                                   │
│                         'cost_details': {                                   │
│                             'upstream_inference_cost': 7.02e-05,            │
│                             'upstream_inference_prompt_cost': 3.6e-05,      │
│                             'upstream_inference_completions_cost': 3.42e-05 │
│                         }                                                   │
│                     },                                                      │
│                     'model_provider': 'openai',                             │
│                     'model_name': 'openai/gpt-oss-120b',                    │
│                     'system_fingerprint': None,                             │
│                     'id': 'gen-1774640370-sqjNjbYVNJO3wkXOLA9c',            │
│                     'finish_reason': 'tool_calls',                          │
│                     'logprobs': None                                        │
│                 },                                                          │
│                 id='lc_run--019d30cf-134e-7eb1-979a-9637a181ea9a-0',        │
│                 tool_calls=[                                                │
│                     {                                                       │
│                         'name': 'bash',                                     │
│                         'args': {'command': 'ls -R'},                       │
│                         'id': 'chatcmpl-tool-b30f45be1e6e1153',             │
│                         'type': 'tool_call'                                 │
│                     }                                                       │
│                 ],                                                          │
│                 invalid_tool_calls=[],                                      │
│

╭──────────────────────────────────────────────── 🤖 LLM Response ────────────────────────────────────────────────╮
│ {                                                                                                               │
│     'tools': {                                                                                                  │
│         'messages': [                                                                                           │
│             ToolMessage(                                                                                        │
│                 content='.:\nagent\nlogs\nmain.py\ntest.ipynb\ntests\n\n./agent:\nconfig.py\nllm.py\nloop.py\nt │
│ ools.py\n\n./logs:\n\n./tests:\n',                                                                              │
│                 name='bash',                                                                                    │
│                 tool_call_id='chatcmpl-tool-b30f45be1e6e1153'                                                   │
│             )                                                                                                   │
│         ]                                                                                                       │
│     }                                                                                                           │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 🤖 LLM Response ──────────────────────────────╮
│ {                                                                           │
│     'agent': {                                                              │
│         'messages': [                                                       │
│             AIMessage(                                                      │
│                 content='',                                                 │
│                 additional_kwargs={'refusal': None},                        │
│                 response_metadata={                                         │
│                     'token_usage': {                                        │
│                         'completion_tokens': 82,                            │
│                         'prompt_tokens': 311,                               │
│                         'total_tokens': 393,                                │
│                         'completion_tokens_details': {                      │
│                             'accepted_prediction_tokens': None,             │
│                             'audio_tokens': 0,                              │
│                             'reasoning_tokens': 61,                         │
│                             'rejected_prediction_tokens': None,             │
│                             'image_tokens': 0                               │
│                         },                                                  │
│                         'prompt_tokens_details': {                          │
│                             'audio_tokens': 0,                              │
│                             'cached_tokens': 64,                            │
│                             'cache_write_tokens': 0,                        │
│                             'video_tokens': 0                               │
│                         },                                                  │
│                         'cost': 9.585e-05,                                  │
│                         'is_byok': False,                                   │
│                         'cost_details': {                                   │
│                             'upstream_inference_cost': 9.585e-05,           │
│                             'upstream_inference_prompt_cost': 4.665e-05,    │
│                             'upstream_inference_completions_cost': 4.92e-05 │
│                         }                                                   │
│                     },                                                      │
│                     'model_provider': 'openai',                             │
│                     'model_name': 'openai/gpt-oss-120b',                    │
│                     'system_fingerprint': None,                             │
│                     'id': 'gen-1774640372-dcmwDKtj7g4C6JrC1vS8',            │
│                     'finish_reason': 'tool_calls',                          │
│                     'logprobs': None                                        │
│                 },                                                          │
│                 id='lc_run--019d30cf-197e-71e0-a4b2-00b1125f8955-0',        │
│                 tool_calls=[                                                │
│                     {                                                       │
│                         'name': 'read_file',                                │
│                         'args': {'path': 'main.py'},                        │
│                         'id': 'chatcmpl-tool-b8c5dc5f5ed53169',             │
│                         'type': 'tool_call'                                 │
│                     }                                                       │
│                 ],                                                          │
│                 invalid_tool_calls=[],                                      │
│

...

╭───────────────────────────── 🤖 LLM Response ──────────────────────────────╮
│ {                                                                          │
│     'agent': {                                                             │
│         'messages': [                                                      │
│             AIMessage(                                                     │
│                 content='',                                                │
│                 additional_kwargs={'refusal': None},                       │
│                 response_metadata={                                        │
│                     'token_usage': {                                       │
│                         'completion_tokens': 110,                          │
│                         'prompt_tokens': 1195,                             │
│                         'total_tokens': 1305,                              │
│                         'completion_tokens_details': {                     │
│                             'accepted_prediction_tokens': None,            │
│                             'audio_tokens': 0,                             │
│                             'reasoning_tokens': 90,                        │
│                             'rejected_prediction_tokens': None,            │
│                             'image_tokens': 0                              │
│                         },                                                 │
│                         'prompt_tokens_details': {                         │
│                             'audio_tokens': 0,                             │
│                             'cached_tokens': 64,                           │
│                             'cache_write_tokens': 0,                       │
│                             'video_tokens': 0                              │
│                         },                                                 │
│                         'cost': 0.00024525,                                │
│                         'is_byok': False,                                  │
│                         'cost_details': {                                  │
│                             'upstream_inference_cost': 0.00024525,         │
│                             'upstream_inference_prompt_cost': 0.00017925,  │
│                             'upstream_inference_completions_cost': 6.6e-05 │
│                         }                                                  │
│                     },                                                     │
│                     'model_provider': 'openai',                            │
│                     'model_name': 'openai/gpt-oss-120b',                   │
│                     'system_fingerprint': None,                            │
│                     'id': 'gen-1774640377-kjSNOyvHhwcUQ5FjUlVA',           │
│                     'finish_reason': 'tool_calls',                         │
│                     'logprobs': None                                       │
│                 },                                                         │
│                 id='lc_run--019d30cf-2f33-7ea0-8be1-33dac729d5cc-0',       │
│                 tool_calls=[                                               │
│                     {                                                      │
│                         'name': 'read_file',                               │
│                         'args': {'path': 'agent/llm.py'},                  │
│                         'id': 'chatcmpl-tool-92048170d1429fe5',            │
│                         'type': 'tool_call'                                │
│                     }                                                      │
│                 ],                                                         │
│                 invalid_tool_calls=[],                                     │
│                 usage_metadata={                 

╭──────────────────────────────────────────────── 🤖 LLM Response ────────────────────────────────────────────────╮
│ {                                                                                                               │
│     'tools': {                                                                                                  │
│         'messages': [                                                                                           │
│             ToolMessage(                                                                                        │
│                 content='from langchain_openai import ChatOpenAI\n\ndef call_agent():\n    llm = ChatOpenAI(\n  │
│ model="openai/gpt-oss-120b",\n    api_key=os.getenv("OPENROUTER_API_KEY"),\n                                    │
│ base_url="https://openrouter.ai/api/v1",\n)',                                                                   │
│                 name='read_file',                                                                               │
│                 tool_call_id='chatcmpl-tool-92048170d1429fe5'                                                   │
│             )                                                                                                   │
│         ]                                                                                                       │
│     }                                                                                                           │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 🤖 LLM Response ───────────────────────────────╮
│ {                                                                            │
│     'agent': {                                                               │
│         'messages': [                                                        │
│             AIMessage(                                                       │
│                 content='',                                                  │
│                 additional_kwargs={'refusal': None},                         │
│                 response_metadata={                                          │
│                     'token_usage': {                                         │
│                         'completion_tokens': 84,                             │
│                         'prompt_tokens': 1292,                               │
│                         'total_tokens': 1376,                                │
│                         'completion_tokens_details': {                       │
│                             'accepted_prediction_tokens': None,              │
│                             'audio_tokens': 0,                               │
│                             'reasoning_tokens': 61,                          │
│                             'rejected_prediction_tokens': None,              │
│                             'image_tokens': 0                                │
│                         },                                                   │
│                         'prompt_tokens_details': {                           │
│                             'audio_tokens': 0,                               │
│                             'cached_tokens': 1120,                           │
│                             'cache_write_tokens': 0,                         │
│                             'video_tokens': 0                                │
│                         },                                                   │
│                         'cost': 0.0002442,                                   │
│                         'is_byok': False,                                    │
│                         'cost_details': {                                    │
│                             'upstream_inference_cost': 0.0002442,            │
│                             'upstream_inference_prompt_cost': 0.0001938,     │
│                             'upstream_inference_completions_cost': 5.04e-05  │
│                         }                                                    │
│                     },                                                       │
│                     'model_provider': 'openai',                              │
│                     'model_name': 'openai/gpt-oss-120b',                     │
│                     'system_fingerprint': None,                              │
│                     'id': 'gen-1774640379-WKcdlhdpb6MX2udxiOtA',             │
│                     'finish_reason': 'tool_calls',                           │
│                     'logprobs': None                                         │
│                 },                                                           │
│                 id='lc_run--019d30cf-36cc-77a1-8975-86c5728ffce0-0',         │
│                 tool_calls=[                                                 │
│                     {                                                        │
│                         'name': 'read_file',                                 │
│                         'args': {'path': 'agent/tools.py'},                  │
│                         'id': 'chatcmpl-tool-bf87a5404f2ae1ed',              │
│                         'type': 'tool_call'                                  │
│                     }                                                        │
│                 ],                                                           │
│                 invalid_tool_c

╭──────────────────────────────────────────────── 🤖 LLM Response ────────────────────────────────────────────────╮
│ {                                                                                                               │
│     'tools': {                                                                                                  │
│         'messages': [                                                                                           │
│             ToolMessage(                                                                                        │
│                 content='import subprocess\nfrom pathlib import Path\n\n@tool\ndef bash(command: str) -> str:\n │
│ """Executes a bash command and returns the output or error."""\n    try:\n        # Use shell=True to allow     │
│ piping and redirections\n        result = subprocess.run(\n            command, \n            shell=True, \n    │
│ capture_output=True, \n            text=True, \n            timeout=30  # Safety timeout\n        )\n        if │
│ result.returncode == 0:\n            return result.stdout\n        return f"Error (Exit Code                    │
│ {result.returncode}): {result.stderr}"\n    except subprocess.TimeoutExpired:\n        return "Error: Command   │
│ timed out after 30 seconds."\n    except Exception as e:\n        return f"Unexpected Error:                    │
│ {str(e)}"\n\n@tool\ndef read_file(path: str) -> str:\n    """Reads and returns the content of a file at the     │
│ specified path."""\n    try:\n        with open(path, \'r\', encoding=\'utf-8\') as f:\n            return      │
│ f.read()\n    except FileNotFoundError:\n        return f"Error: The file at {path} was not found."\n    except │
│ Exception as e:\n        return f"Error reading file: {str(e)}"\n    \n@tool\ndef write_file(path: str,         │
│ content: str) -> str:\n    """Writes content to a file, creating parent directories if they don\'t exist."""\n  │
│ try:\n        file_path = Path(path)\n        # Create directories if they are missing (similar to \'mkdir      │
│ -p\')\n        file_path.parent.mkdir(parents=True, exist_ok=True)\n        \n                                  │
│ file_path.write_text(content, encoding=\'utf-8\')\n        return f"Successfully wrote {len(content)}           │
│ characters to {path}"\n    except Exception as e:\n        return f"Error writing to {path}: {str(e)}"\n\ndef   │
│ execute_tool(tool_name: str, args: dict):\n    if tool_name == "bash":\n        return bash(args["command"])\n  │
│ elif tool_name == "read_file":\n        return read_file(args["path"])\n    elif tool_name == "write_file":\n   │
│ return write_file(args["path"], args["content"])\n    else:\n        return f"Unknown tool: {tool_name}"\n',    │
│                 name='read_file',                                                                               │
│                 tool_call_id='chatcmpl-tool-bf87a5404f2ae1ed'                                                   │
│             )                                                                                                   │
│         ]                                                                                                       │
│     }                                                                                                           │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🤖 LLM Response ────────────────────────────────────────────────╮
│ {                                                                                                               │
│     'agent': {                                                                                                  │
│         'messages': [                                                                                           │
│             AIMessage(                                                                                          │
│                 content='I’m not sure which file you’d like to see. Could you let me know the name (or path) of │
│ the file in the current directory that you’d like me to read?',                                                 │
│                 additional_kwargs={'refusal': None},                                                            │
│                 response_metadata={                                                                             │
│                     'token_usage': {                                                                            │
│                         'completion_tokens': 200,                                                               │
│                         'prompt_tokens': 1777,                                                                  │
│                         'total_tokens': 1977,                                                                   │
│                         'completion_tokens_details': {                                                          │
│                             'accepted_prediction_tokens': None,                                                 │
│                             'audio_tokens': 0,                                                                  │
│                             'reasoning_tokens': 174,                                                            │
│                             'rejected_prediction_tokens': None,                                                 │
│                             'image_tokens': 0                                                                   │
│                         },                                                                                      │
│                         'prompt_tokens_details': {                                                              │
│                             'audio_tokens': 0,                                                                  │
│                             'cached_tokens': 64,                                                                │
│                             'cache_write_tokens': 0,                                                            │
│                             'video_tokens': 0                                                                   │
│                         },                                                                                      │
│                         'cost': 0.00038655,                                                                     │
│                         'is_byok': False,                                                                       │
│                         'cost_details': {                                                                       │
│                             'upstream_inference_cost': 0.00038655,                                              │
│                             'upstream_inference_prompt_cost': 0.00026655,                                       │
│                             'upstream_inference_completions_cost': 0.00012                                      │
│                         }                                                                                       │
│                     },                                                                                          │
│                     'model_provider': 'openai',        

In [19]:
from e2b_code_interpreter import Sandbox, SandboxQuery, SandboxState

sbx = Sandbox.create(api_key = os.getenv("E2B_API_KEY")) # Creates a persistent sandbox session
execution = sbx.run_code("print('hello world')") # Execute Python inside the sandbox
print(execution.logs)

Logs(stdout: ['hello world\n'], stderr: [])


In [23]:
paginator = Sandbox.list()

# Get the first page of paused sandboxes
sandboxes = paginator.next_items()

In [24]:
sandboxes

[SandboxInfo(sandbox_id='iztm80o393c66ttjpnbnd', sandbox_domain=None, template_id='nlhz8vlwyupq845jsdg9', name='code-interpreter-v1', metadata={}, started_at=datetime.datetime(2026, 3, 28, 7, 27, 13, 273449, tzinfo=tzutc()), end_at=datetime.datetime(2026, 3, 28, 7, 32, 13, 273449, tzinfo=tzutc()), state=<SandboxState.RUNNING: 'running'>, cpu_count=2, memory_mb=2048, envd_version='0.5.8', _envd_access_token=None, allow_internet_access=None, network=None, lifecycle=None, volume_mounts=[])]

In [14]:
# Read local file relative to the current working directory
with open("./hello.py", "rb") as file:
   # Upload file to the sandbox to absolute path '/home/user/my-file'
	sbx.files.write("/home/user/hello.py", file)

In [15]:
file_content = sbx.files.read('/home/user/hello.py')

In [16]:
file_content

"print('Hello, World!')"

In [17]:
result = sbx.commands.run('python /home/user/hello.py')

In [18]:
print(result)

CommandResult(stderr='', stdout='Hello, World!\n', exit_code=0, error='')


In [28]:
from e2b_code_interpreter import Sandbox, SandboxQuery, SandboxState
import os

class SandBox:
    """Sandbox for safe execution"""
    def __init__(self):
        self.sandbox = Sandbox.create(
            api_key = os.getenv("E2B_API_KEY"), 
            timeout = 10 * 60,
            lifecycle={
                "on_timeout": "pause"
            },
        )
        print('Sandbox created', self.sandbox.sandbox_id)
    
    def pause_sandbox(self):
        self.sandbox.pause() 
        print('Sandbox paused', self.sandbox.sandbox_id)

    def resume_sandbox(self):
        self.sandbox.connect()
        print('Connected to the sandbox', self.sandbox.sandbox_id)

    def close_sandbox(self):
        self.sandbox.kill()

In [ ]:
sbx = SandBox()
sbx.sandbox.sandbox_id


Sandbox created icbt0uzalhdn1kb6ez1yg


'icbt0uzalhdn1kb6ez1yg'

In [36]:
paginator = Sandbox.list()

# Get the first page of paused sandboxes
sandboxes = paginator.next_items()
sandboxes

[]

In [35]:
Sandbox.kill('i0xblladeeb9vrg6z34t9')

True

In [7]:
from playwright.async_api import async_playwright
import nest_asyncio
nest_asyncio.apply()

# In Jupyter, you can use await directly at the top level
pw = await async_playwright().start()
browser = await pw.chromium.launch(headless=False)
page = await browser.new_page()

await page.goto("https://playwright.dev")
print(await page.title())

# Always remember to close to free resources
await browser.close()
await pw.stop()


NotImplementedError: 